# Воркшоп №3 — Fusion-стратегии

**Курсовая:** «Использование мультимодальных эмбеддингов в рекомендательных системах».

У меня есть эмбеддинги для **14 787** треков. В этом ноутбуке объединяю три модальности тремя способами:

1. **Early fusion** — конкатенация векторов
2. **Late fusion** — взвешенная сумма cosine similarity
3. **Cross-attention** — маленький Transformer на 3 токенах

На sanity-check из воркшопа №2: TEXT Recall@10 ≈ 0.08, IMAGE ≈ 0.075, AUDIO ≈ 0.025 — все выше random (0.0047). Значит, fusion имеет смысл.

Считаю на CPU, чтобы не перегревать Mac.

## 0. Установка зависимостей

In [1]:
%pip install -q torch numpy pandas tqdm pyarrow scikit-learn

Note: you may need to restart the kernel to use updated packages.


## 1. Конфигурация

In [2]:
from __future__ import annotations

import json
import logging
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("fusion")

DEVICE = "cpu"          # "mps" или "cuda" если готов к нагреву
BATCH_SIZE = 32         # для cross-attention обучения
TRAIN_EPOCHS = 2        # мало эпох = быстро + не греет
MAX_TRAIN_SRC = 3000    # обучаем fusion только на подвыборке рёбер
NEGATIVES = 5           # негативов на позитив в contrastive loss
FUSION_DIM = 256        # общая размерность для cross-attention
K_EVAL = 10             # Recall@K для sanity-check

DATA_DIR = Path("data")
EMB_DIR = DATA_DIR / "embeddings"
FUSION_DIR = DATA_DIR / "fusion"
FUSION_DIR.mkdir(parents=True, exist_ok=True)

log.info("Device: %s", DEVICE)

2026-06-10 02:04:57,981 | INFO | Device: cpu


## 2. Загрузка эмбеддингов

In [3]:
def l2norm(x: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(x, axis=1, keepdims=True)
    return x / np.clip(norms, 1e-12, None)


text_npz = np.load(EMB_DIR / "text.npz")
image_npz = np.load(EMB_DIR / "image.npz")
audio_npz = np.load(EMB_DIR / "audio.npz")

TRACK_IDS = text_npz["track_ids"].astype(np.int64)
text_emb = l2norm(text_npz["emb"].astype(np.float32))
image_emb = l2norm(image_npz["emb"].astype(np.float32))
audio_emb = l2norm(audio_npz["emb"].astype(np.float32))

assert np.array_equal(TRACK_IDS, image_npz["track_ids"])
assert np.array_equal(TRACK_IDS, audio_npz["track_ids"])

N = len(TRACK_IDS)
tid_to_idx = {int(t): i for i, t in enumerate(TRACK_IDS)}
log.info("Loaded %d tracks: text=%s image=%s audio=%s", N, text_emb.shape, image_emb.shape, audio_emb.shape)

# Ground truth из edges.parquet
edges_df = pd.read_parquet(DATA_DIR / "edges.parquet")
edges_in = edges_df.loc[edges_df["in_corpus"] & edges_df["dst_track_id"].notna()].copy()
edges_in["dst_track_id"] = edges_in["dst_track_id"].astype(np.int64)
edges_in = edges_in.loc[
    edges_in["src_track_id"].isin(tid_to_idx) & edges_in["dst_track_id"].isin(tid_to_idx)
]

gt_by_src: dict[int, set[int]] = {}
for src_tid, grp in edges_in.sort_values("match", ascending=False).groupby("src_track_id"):
    top = grp.head(K_EVAL)["dst_track_id"].astype(int).tolist()
    if top:
        gt_by_src[int(src_tid)] = set(top)

log.info("Ground truth: %d src tracks with top-%d neighbors", len(gt_by_src), K_EVAL)

2026-06-10 02:04:58,284 | INFO | Loaded 14787 tracks: text=(14787, 768) image=(14787, 512) audio=(14787, 512)


2026-06-10 02:04:59,685 | INFO | Ground truth: 7606 src tracks with top-10 neighbors


## 3. Метрика Recall@K

In [4]:
def recall_at_k_vector(emb: np.ndarray, gt_by_src: dict[int, set[int]], k: int = K_EVAL) -> float:
    """Recall@K для одного эмбеддинг-пространства."""
    hits = total = 0
    src_idxs = [tid_to_idx[s] for s in gt_by_src if s in tid_to_idx]
    chunk = 256
    for start in range(0, len(src_idxs), chunk):
        block = src_idxs[start : start + chunk]
        sims = emb[block] @ emb.T
        for j, i in enumerate(block):
            sims[j, i] = -np.inf
        top_idx = np.argpartition(-sims, k, axis=1)[:, :k]
        for row_j, src_i in enumerate(block):
            src_tid = int(TRACK_IDS[src_i])
            gt = gt_by_src[src_tid]
            pred_tids = {int(TRACK_IDS[x]) for x in top_idx[row_j]}
            hits += len(pred_tids & gt) / max(len(gt), 1)
            total += 1
    return hits / max(total, 1)


def recall_at_k_late(
    gt_by_src: dict[int, set[int]],
    weights: tuple[float, float, float],
    k: int = K_EVAL,
) -> float:
    """Recall@K для late fusion: w_text*sim_text + w_image*sim_image + w_audio*sim_audio."""
    w_t, w_i, w_a = weights
    hits = total = 0
    src_idxs = [tid_to_idx[s] for s in gt_by_src if s in tid_to_idx]
    chunk = 128
    for start in range(0, len(src_idxs), chunk):
        block = src_idxs[start : start + chunk]
        sims = (
            w_t * (text_emb[block] @ text_emb.T)
            + w_i * (image_emb[block] @ image_emb.T)
            + w_a * (audio_emb[block] @ audio_emb.T)
        )
        for j, i in enumerate(block):
            sims[j, i] = -np.inf
        top_idx = np.argpartition(-sims, k, axis=1)[:, :k]
        for row_j, src_i in enumerate(block):
            src_tid = int(TRACK_IDS[src_i])
            gt = gt_by_src[src_tid]
            pred_tids = {int(TRACK_IDS[x]) for x in top_idx[row_j]}
            hits += len(pred_tids & gt) / max(len(gt), 1)
            total += 1
    return hits / max(total, 1)


def random_recall(gt_by_src: dict[int, set[int]], k: int = K_EVAL) -> float:
    eps = [min(k * len(gt) / (N - 1), 1.0) for gt in gt_by_src.values()]
    return float(np.mean(eps))


print("Baseline (из воркшопа №2):")
print(f"  TEXT  Recall@{K_EVAL}: {recall_at_k_vector(text_emb, gt_by_src):.3f}")
print(f"  IMAGE Recall@{K_EVAL}: {recall_at_k_vector(image_emb, gt_by_src):.3f}")
print(f"  AUDIO Recall@{K_EVAL}: {recall_at_k_vector(audio_emb, gt_by_src):.3f}")
print(f"  RANDOM Recall@{K_EVAL}: {random_recall(gt_by_src):.4f}")

Baseline (из воркшопа №2):


  TEXT  Recall@10: 0.080


  IMAGE Recall@10: 0.075


  AUDIO Recall@10: 0.025
  RANDOM Recall@10: 0.0047


## 4. Early Fusion

In [5]:
EARLY_PATH = FUSION_DIR / "early_fusion.npz"

if EARLY_PATH.exists():
    log.info("Early fusion already exists — loading.")
    early_emb = np.load(EARLY_PATH)["emb"]
else:
    t0 = time.time()
    early_emb = l2norm(np.concatenate([text_emb, image_emb, audio_emb], axis=1))
    np.savez_compressed(EARLY_PATH, track_ids=TRACK_IDS, emb=early_emb.astype(np.float32))
    log.info("Early fusion saved: shape=%s, %.1fs", early_emb.shape, time.time() - t0)

early_recall = recall_at_k_vector(early_emb, gt_by_src)
print(f"Early fusion Recall@{K_EVAL}: {early_recall:.3f}")
print(f"  vs TEXT-only:  {recall_at_k_vector(text_emb, gt_by_src):.3f}")
print(f"  vs IMAGE-only: {recall_at_k_vector(image_emb, gt_by_src):.3f}")
print(f"  vs AUDIO-only: {recall_at_k_vector(audio_emb, gt_by_src):.3f}")

2026-06-10 02:05:06,190 | INFO | Early fusion saved: shape=(14787, 1792), 3.1s


Early fusion Recall@10: 0.096


  vs TEXT-only:  0.080


  vs IMAGE-only: 0.075


  vs AUDIO-only: 0.025


## 5. Late Fusion

In [6]:
LATE_WEIGHTS_PATH = FUSION_DIR / "late_fusion_weights.json"

if LATE_WEIGHTS_PATH.exists():
    best_weights = tuple(json.loads(LATE_WEIGHTS_PATH.read_text())["weights"])
    log.info("Late fusion weights loaded: %s", best_weights)
else:
    # Подвыборка для быстрого grid search (не греет CPU)
    rng = np.random.default_rng(42)
    eval_src = list(gt_by_src.keys())
    if len(eval_src) > 1500:
        eval_src = rng.choice(eval_src, size=1500, replace=False).tolist()
    eval_gt = {s: gt_by_src[s] for s in eval_src}

    grid = [0.0, 0.25, 0.5, 0.75, 1.0]
    best_score = -1.0
    best_weights = (1/3, 1/3, 1/3)

    candidates = []
    for w_t in grid:
        for w_i in grid:
            for w_a in grid:
                if w_t + w_i + w_a == 0:
                    continue
                s = w_t + w_i + w_a
                candidates.append((w_t / s, w_i / s, w_a / s))

    for w in tqdm(candidates, desc="late fusion grid"):
        score = recall_at_k_late(eval_gt, w)
        if score > best_score:
            best_score = score
            best_weights = w

    LATE_WEIGHTS_PATH.write_text(json.dumps({
        "weights": list(best_weights),
        "eval_recall_at_10_on_subset": best_score,
        "note": "late fusion has no single vector; weights used at scoring time",
    }, indent=2))
    log.info("Best late weights: %s (subset Recall@10=%.3f)", best_weights, best_score)

late_recall = recall_at_k_late(gt_by_src, best_weights)
print(f"Late fusion Recall@{K_EVAL}: {late_recall:.3f}")
print(f"  weights: text={best_weights[0]:.2f}, image={best_weights[1]:.2f}, audio={best_weights[2]:.2f}")

late fusion grid:   0%|          | 0/124 [00:00<?, ?it/s]

2026-06-10 02:05:57,394 | INFO | Best late weights: (0.6666666666666666, 0.16666666666666666, 0.16666666666666666) (subset Recall@10=0.110)


Late fusion Recall@10: 0.105
  weights: text=0.67, image=0.17, audio=0.17


## 6. Cross-Attention Fusion

In [7]:
CROSS_PATH = FUSION_DIR / "cross_attention.npz"
CROSS_MODEL_PATH = FUSION_DIR / "cross_attention.pt"


class CrossModalFusion(nn.Module):
    def __init__(self, d_text: int = 768, d_image: int = 512, d_audio: int = 512, d_model: int = FUSION_DIM):
        super().__init__()
        self.proj_t = nn.Linear(d_text, d_model)
        self.proj_i = nn.Linear(d_image, d_model)
        self.proj_a = nn.Linear(d_audio, d_model)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=4, dim_feedforward=512,
            batch_first=True, dropout=0.1,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=1)

    def forward(self, t: torch.Tensor, i: torch.Tensor, a: torch.Tensor) -> torch.Tensor:
        tokens = torch.stack([self.proj_t(t), self.proj_i(i), self.proj_a(a)], dim=1)
        out = self.encoder(tokens).mean(dim=1)
        return F.normalize(out, p=2, dim=1)


def _to_tensor(emb: np.ndarray, idx: np.ndarray) -> torch.Tensor:
    return torch.from_numpy(emb[idx]).to(DEVICE)


def build_train_pairs(max_src: int = MAX_TRAIN_SRC) -> list[tuple[int, int]]:
    rng = np.random.default_rng(42)
    src_list = list(gt_by_src.keys())
    if len(src_list) > max_src:
        src_list = rng.choice(src_list, size=max_src, replace=False).tolist()
    pairs = []
    for src_tid in src_list:
        dsts = [d for d in gt_by_src[src_tid] if d in tid_to_idx and d != src_tid]
        if not dsts:
            continue
        dst_tid = int(rng.choice(dsts))
        pairs.append((tid_to_idx[src_tid], tid_to_idx[dst_tid]))
    return pairs


@torch.no_grad()
def encode_all(model: CrossModalFusion, batch_size: int = BATCH_SIZE) -> np.ndarray:
    model.eval()
    out = []
    for start in tqdm(range(0, N, batch_size), desc="cross-attn encode"):
        idx = np.arange(start, min(start + batch_size, N))
        t = _to_tensor(text_emb, idx)
        i = _to_tensor(image_emb, idx)
        a = _to_tensor(audio_emb, idx)
        out.append(model(t, i, a).cpu().numpy())
    return np.concatenate(out, axis=0).astype(np.float32)


if CROSS_PATH.exists() and CROSS_MODEL_PATH.exists():
    log.info("Cross-attention fusion already exists — loading.")
    model = CrossModalFusion().to(DEVICE)
    model.load_state_dict(torch.load(CROSS_MODEL_PATH, map_location=DEVICE))
    cross_emb = np.load(CROSS_PATH)["emb"]
else:
    pairs = build_train_pairs()
    log.info("Training cross-attention on %d pairs, %d epochs", len(pairs), TRAIN_EPOCHS)

    model = CrossModalFusion().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    temperature = 0.07

    rng = np.random.default_rng(42)
    for epoch in range(TRAIN_EPOCHS):
        model.train()
        order = rng.permutation(len(pairs))
        losses = []
        for start in tqdm(range(0, len(order), BATCH_SIZE), desc=f"epoch {epoch+1}"):
            batch_pairs = [pairs[i] for i in order[start : start + BATCH_SIZE]]
            src_idx = np.array([p[0] for p in batch_pairs])
            dst_idx = np.array([p[1] for p in batch_pairs])

            src_t = _to_tensor(text_emb, src_idx)
            src_i = _to_tensor(image_emb, src_idx)
            src_a = _to_tensor(audio_emb, src_idx)
            dst_t = _to_tensor(text_emb, dst_idx)
            dst_i = _to_tensor(image_emb, dst_idx)
            dst_a = _to_tensor(audio_emb, dst_idx)

            anchor = model(src_t, src_i, src_a)
            positive = model(dst_t, dst_i, dst_a)
            logits = (anchor @ positive.T) / temperature
            labels = torch.arange(len(batch_pairs), device=DEVICE)
            loss = F.cross_entropy(logits, labels)

            opt.zero_grad()
            loss.backward()
            opt.step()
            losses.append(loss.item())

        log.info("Epoch %d: loss=%.4f", epoch + 1, np.mean(losses))

    torch.save(model.state_dict(), CROSS_MODEL_PATH)
    cross_emb = encode_all(model)
    np.savez_compressed(CROSS_PATH, track_ids=TRACK_IDS, emb=cross_emb)
    log.info("Cross-attention saved: %s", CROSS_PATH)

cross_recall = recall_at_k_vector(cross_emb, gt_by_src)
print(f"Cross-attention Recall@{K_EVAL}: {cross_recall:.3f}")

2026-06-10 02:05:59,310 | INFO | Training cross-attention on 3000 pairs, 2 epochs


epoch 1:   0%|          | 0/94 [00:00<?, ?it/s]

2026-06-10 02:06:00,970 | INFO | Epoch 1: loss=1.8076


epoch 2:   0%|          | 0/94 [00:00<?, ?it/s]

2026-06-10 02:06:01,861 | INFO | Epoch 2: loss=1.2092


cross-attn encode:   0%|          | 0/463 [00:00<?, ?it/s]

2026-06-10 02:06:02,761 | INFO | Cross-attention saved: data/fusion/cross_attention.npz


Cross-attention Recall@10: 0.105


## 7. Сравнение методов

In [8]:
results = pd.DataFrame([
    {"method": "RANDOM",              "recall_at_10": random_recall(gt_by_src), "type": "baseline"},
    {"method": "TEXT-only",           "recall_at_10": recall_at_k_vector(text_emb, gt_by_src),  "type": "unimodal"},
    {"method": "IMAGE-only",          "recall_at_10": recall_at_k_vector(image_emb, gt_by_src), "type": "unimodal"},
    {"method": "AUDIO-only",          "recall_at_10": recall_at_k_vector(audio_emb, gt_by_src), "type": "unimodal"},
    {"method": "Early fusion",        "recall_at_10": early_recall,                          "type": "fusion"},
    {"method": "Late fusion",         "recall_at_10": late_recall,                           "type": "fusion"},
    {"method": "Cross-attention",     "recall_at_10": cross_recall,                          "type": "fusion"},
]).sort_values("recall_at_10", ascending=False).reset_index(drop=True)

results["recall_at_10"] = results["recall_at_10"].round(3)
print(results.to_string(index=False))

RESULTS_PATH = FUSION_DIR / "fusion_comparison.csv"
results.to_csv(RESULTS_PATH, index=False)
print(f"\nSaved to {RESULTS_PATH}")

         method  recall_at_10     type
Cross-attention         0.105   fusion
    Late fusion         0.105   fusion
   Early fusion         0.096   fusion
      TEXT-only         0.080 unimodal
     IMAGE-only         0.075 unimodal
     AUDIO-only         0.025 unimodal
         RANDOM         0.005 baseline

Saved to data/fusion/fusion_comparison.csv


## 8. Итог

Сохранил fusion-артефакты в `data/fusion/`. Дальше — `04_recsys_eval.ipynb`.